# Guardian Candidate Model v1 — First Production Training (Sprint 20)

Runs the exact Sprint 20 spec end to end on a Colab **T4 GPU**, using the official Apache-2.0 YOLOX integration from Sprint 19.1 (`OfficialYoloxTrainer` — nothing here knows YOLOX's internals):

**Dataset → Official YOLOX Training → Evaluate → Error Analysis → Qualitative Report → ONNX Export → Benchmark → COCO-Pretrained Comparison → Candidate**

This notebook expects two zips already uploaded to Google Drive — `guardian-ai.zip` (source only) and `guardian-dataset-v1.zip` (the published dataset). Build them with `scripts/package-colab.sh` and `scripts/package-dataset.sh`; see [`docs/COLAB_SETUP.md`](../../docs/COLAB_SETUP.md) for the full walkthrough. The repository itself never carries dataset bytes — datasets are external assets, resolved everywhere through `GUARDIAN_DATASET_ROOT`.

Rules that still apply here:
- Data comes **only** from the Guardian Dataset Registry (`guardian-fall-detection-v1@1.0.0`, published in Sprint 18) — never raw datasets, never modified.
- `python -m guardian_ai.train` is the same CLI used locally; this notebook adds no training logic of its own. No architecture changes, no detector changes — Sprint 19.1's `DetectorFamily` wrapping is used as-is.
- This produces a **candidate model only**. Nothing here installs into any model zoo or touches the Edge Box — promotion is a separate, later, human decision.

> Hardware: Colab → Runtime → Change runtime type → **T4 GPU**.

## 1. Mount Google Drive

Drive holds the two zips built by `scripts/package-colab.sh` and `scripts/package-dataset.sh` — upload them once directly to the root of `MyDrive`:

```
MyDrive/
  guardian-ai.zip
  guardian-dataset-v1.zip
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"  # guardian-ai.zip + guardian-dataset-v1.zip live here
DRIVE_OUT = "/content/drive/MyDrive/guardian-ai-models/model-v1"
!mkdir -p {DRIVE_OUT}

## 2. Extract the source and install dependencies

`guardian-ai.zip` is this repository's tracked source only — no `.git`, no datasets, no venv/caches, no prior reports/runs (`scripts/package-colab.sh` strips all of that). Extraction overwrites `/content/guardian-ai` every run, so re-running this cell after a disconnect is always safe.

Installation uses **uv**, not plain `pip install -e ai`. `guardian_ai` depends on the real, unmodified upstream YOLOX (Sprint 19.1) pinned to an exact git commit, and getting it installed requires two `[tool.uv]` settings (a per-package build-isolation exception, since its `setup.py` needs torch importable at build time; and a dependency-version override) that plain pip has no equivalent for — `pip install -e ai` reliably fails with `AssertionError: torch is required for pre-compiling ops`. `uv sync` is exactly what local development already uses, so this is the same install path everywhere, not a Colab-specific workaround.

In [ ]:
import os
from pathlib import Path

WORK_DIR = "/content/guardian-ai"
SOURCE_ZIP = f"{DRIVE_DIR}/guardian-ai.zip"
if not Path(SOURCE_ZIP).is_file():
    raise RuntimeError(
        f"{SOURCE_ZIP} not found. Upload guardian-ai.zip (built with "
        "scripts/package-colab.sh) to the root of MyDrive — see docs/COLAB_SETUP.md."
    )

os.makedirs(WORK_DIR, exist_ok=True)
!unzip -oq {SOURCE_ZIP} -d {WORK_DIR}
if not (Path(WORK_DIR) / "ai" / "pyproject.toml").is_file():
    raise RuntimeError(
        f"extraction of {SOURCE_ZIP} into {WORK_DIR} did not produce ai/pyproject.toml "
        "— the zip may be corrupt or built incorrectly. Rebuild it with "
        "scripts/package-colab.sh and re-upload."
    )

%cd {WORK_DIR}
!pip install -q uv
!uv sync --project ai

VENV_BIN = f"{WORK_DIR}/.venv/bin"  # uv workspace venv lives at the repo root
if not (Path(VENV_BIN) / "python").is_file():
    raise RuntimeError(f"uv sync did not produce {VENV_BIN}/python — install failed.")

# Prepend the venv so every later !python / !pip cell (training, evaluate,
# export, ...) transparently uses it, with no need to prefix each one with
# `uv run --project ai`.
os.environ["PATH"] = f"{VENV_BIN}:" + os.environ["PATH"]

## 3. Extract the dataset, set `GUARDIAN_DATASET_ROOT`, validate the workspace

Datasets are external assets — never a hardcoded repo-relative path (`docs/COLAB_SETUP.md`). `guardian-dataset-v1.zip` is extracted to `/content/datasets` on local Colab disk (fast, ephemeral storage) rather than read repeatedly from Drive, and `GUARDIAN_DATASET_ROOT` is set to that path automatically — no manual step, no manual `%env`/`export`.

Every failure surfaces its real cause, in order, so a bad run stops at the first broken thing rather than deep inside `train`:
- **unzip exit code is checked** — an 8.5 GB zip that was still syncing to Drive, or that overran Colab's disk, extracts partially; `unzip -q` hides that, so we inspect its return code and abort with a clear message.
- **structure is checked on disk** — `registry/` and the version's `checksums.json` must exist.
- **checksums are re-verified** — `guardian_ai` lives in the `uv` venv (previous cell), not this kernel's Python, so validation runs as a subprocess of the venv's interpreter with its output **captured and re-raised**, so the underlying error (e.g. a checksum mismatch) is shown directly instead of an opaque `CalledProcessError`.

In [ ]:
import os
import subprocess
from pathlib import Path

DATASET_ZIP = f"{DRIVE_DIR}/guardian-dataset-v1.zip"
GUARDIAN_DATASET_ROOT = "/content/datasets"

if not Path(DATASET_ZIP).is_file():
    raise RuntimeError(
        f"{DATASET_ZIP} not found. Upload guardian-dataset-v1.zip (built with "
        "scripts/package-dataset.sh) to the root of MyDrive — see docs/COLAB_SETUP.md."
    )

os.makedirs(GUARDIAN_DATASET_ROOT, exist_ok=True)
unzip = subprocess.run(  # noqa: S603
    ["unzip", "-oq", DATASET_ZIP, "-d", GUARDIAN_DATASET_ROOT],  # noqa: S607
    capture_output=True,
    text=True,
)
if unzip.returncode != 0:
    print(unzip.stderr)
    raise RuntimeError(
        f"unzip of {DATASET_ZIP} exited {unzip.returncode} — the zip is likely "
        "truncated or corrupt. The usual Colab causes: the Drive upload had not "
        "finished syncing, or /content ran out of disk. Confirm the uploaded "
        "zip's size matches the local file, check disk with `!df -h /content`, "
        "and retry."
    )

# Automatic — no manual %env/export step. Every later cell (training
# included) inherits this, since ! shell commands and subprocess.run both
# run as subprocesses of this kernel.
os.environ["GUARDIAN_DATASET_ROOT"] = GUARDIAN_DATASET_ROOT

registry_dir = Path(GUARDIAN_DATASET_ROOT) / "registry"
version_dir = registry_dir / "guardian-fall-detection-v1" / "1.0.0"
if not registry_dir.is_dir():
    raise RuntimeError(
        f"{registry_dir} does not exist after extracting {DATASET_ZIP}. "
        "The zip may be corrupt, built incorrectly, or from the wrong path — "
        "rebuild it with scripts/package-dataset.sh and re-upload."
    )
if not (version_dir / "checksums.json").is_file():
    raise RuntimeError(
        f"{version_dir}/checksums.json is missing after extraction — the zip is "
        "incomplete (likely truncated) or built from the wrong path. Rebuild "
        "with scripts/package-dataset.sh and re-upload the full file."
    )

# guardian_ai lives in the uv venv (previous cell), not this notebook
# kernel's own Python. Validate through a subprocess of the venv's
# interpreter, capturing its output so its REAL error (e.g. a checksum
# mismatch from an incomplete extraction) is shown directly rather than
# wrapped in an opaque CalledProcessError.
validate_script = Path("/content/_validate_dataset.py")
validate_script.write_text(
    "from pathlib import Path\n"
    "from guardian_ai.acquisition.registry import VideoDatasetRegistry\n"
    f"registry_dir = Path({str(registry_dir)!r})\n"
    "published = VideoDatasetRegistry(registry_dir).get("
    "'guardian-fall-detection-v1', '1.0.0')\n"
    "print('workspace validated:', published)\n"
)
result = subprocess.run(  # noqa: S603 - fixed argv, no user input
    [f"{VENV_BIN}/python", str(validate_script)],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(
        "dataset workspace validation failed — the registry could not be "
        "verified against its recorded checksums. This almost always means an "
        "incomplete or corrupt extraction (an unfinished Drive upload, or Colab "
        "running out of disk). Underlying error:\n\n" + result.stderr
    )
print("GUARDIAN_DATASET_ROOT =", GUARDIAN_DATASET_ROOT)

## 4. Train — the exact Sprint 20 hyperparameters

`ai/training/configs/model-v1.yaml`: official YOLOX-Tiny (Apache-2.0, Sprint 19.1), 640px, COCO-pretrained checkpoint (auto-downloaded, checksum-pinned), 30 epochs, batch 16, workers 2, seed 42, SGD lr=0.01 with **5-epoch linear warmup then cosine annealing** (fixes the numerical divergence Sprint 19.1's comparison report disclosed), early stopping patience 10, mixed precision on, `device: cuda` (the T4). The config has no `dataset.registry_root` — it resolves through `GUARDIAN_DATASET_ROOT`, set in the previous cell. One command, one YAML — the config is the complete, reproducible recipe.

In [ ]:
!python -m guardian_ai.train train --config ai/training/configs/model-v1.yaml

import pathlib

RUN = sorted(pathlib.Path("ai/training/runs").iterdir())[-1]
print("run:", RUN)

If Colab disconnects mid-training, reconnect, re-run the mount/extract/install/dataset cells above (1–3), then resume from the last checkpoint (optimizer/scheduler/epoch/early-stopping state all restored):

```
!python -m guardian_ai.train resume --run {RUN}
```

## 5. Evaluate + visual reports

The full mandated metric set — precision, recall, F1, mAP@50, mAP@50-95, per-class, confusion matrix, absolute FP/FN — plus confusion matrix / PR curve / loss curve / summary PDF.

In [ ]:
!python -m guardian_ai.train evaluate --run {RUN}
!python -m guardian_ai.train report --run {RUN}

from IPython.display import Image as ShowImage
from IPython.display import display

display(ShowImage(filename=f"{RUN}/reports/loss_curve.png"))
display(ShowImage(filename=f"{RUN}/reports/confusion_matrix.png"))

## 6. Error analysis

Top false-positive images, top false-negative images, the most confident wrong detections, the worst-localized correct detections, and the most confused class pairs — written to `reports/error-analysis.json`.

In [ ]:
!python -m guardian_ai.train error-analysis --run {RUN} --top-k 10

## 7. Qualitative report

50 random validation predictions, ground truth (green) and predictions (cyan, with confidence) drawn on the same letterboxed frame the model saw.

In [ ]:
!python -m guardian_ai.train qualitative --run {RUN} --count 50

import glob

samples = sorted(glob.glob(f"{RUN}/reports/qualitative/*.png"))[:4]
for path in samples:
    display(ShowImage(filename=path))

## 8. ONNX export (validated) + benchmark

Export runs the ONNX structural checker **and** a torch-vs-onnxruntime parity check (max |Δ| ≤ 1e-4) — a diverging artifact is rejected before any manifest exists. Benchmark measures latency/memory/size on ONNX Runtime CPU — these numbers become the baseline for every future run.

In [ ]:
!python -m guardian_ai.train export --run {RUN} --version 1.0.0 --name guardian-fall-v1
!python -m guardian_ai.train benchmark --run {RUN} --runs 50

## 9. COCO-pretrained comparison — PROMOTE or KEEP COCO

Fetches the official Apache-2.0 YOLOX-Tiny COCO checkpoint (pinned SHA-256, ADR-0003) through the existing edge/ installer, evaluates it on the *same* test images with our own harness (person class only), and compares precision/recall/false-positives/latency/memory against this run.

In [ ]:
!python -m guardian_ai.train coco-compare --run {RUN} \
    --zoo-root /content/coco-baseline-zoo \
    --edge-project-root /content/guardian-ai/edge \
    --runs 50

import json

comparison = json.loads((RUN / "coco-comparison.json").read_text())
print("VERDICT:", comparison["verdict"])
for reason in comparison["reasons"]:
    print(" -", reason)

## 10. Mark as candidate — never deploy from here

This is a **candidate model only**. It is never installed into any model zoo from this notebook; promotion is a separate, later, human decision made after reviewing everything above (see `reports/model-v1/README.md` in the repo for the write-up template).

In [ ]:
!python -m guardian_ai.train candidate --run {RUN} \
    --notes "Full Colab T4 run: 19,940 train images, 30 epochs, COCO-pretrained, \
5-epoch warmup + cosine, mixed precision"

!cp {RUN}/export/model.onnx {RUN}/export/manifest.json \
    {RUN}/export/guardian-validation.json {DRIVE_OUT}/
!cp {RUN}/reports/evaluation.json {RUN}/reports/error-analysis.json \
    {RUN}/coco-comparison.json {DRIVE_OUT}/
print("Artifacts + reports copied to", DRIVE_OUT)
print("Candidate only. No zoo install.")
print("Hand this run directory to review for the promotion decision.")